In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# 1. 加载数据集
diabetes = load_diabetes()
X = diabetes.data  # 特征 (442, 10)
y = diabetes.target  # 目标值 (442,)

# 2. 数据预处理
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)  # 标准化特征
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()  # 标准化目标值

# 将数据转换为 PyTorch 张量
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

# 创建数据加载器
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# 3. 定义神经网络模型
model = nn.Sequential(
    nn.Linear(10, 64),  # 输入层到隐藏层，10个特征输入，64个神经元
    nn.ReLU(),          # 激活函数
    nn.Linear(64, 32),  # 隐藏层到隐藏层，64个输入，32个神经元
    nn.ReLU(),          # 激活函数
    nn.Linear(32, 1)    # 输出层，1个输出（糖尿病进展）
)

# 4. 定义损失函数和优化器
criterion = nn.MSELoss()  # 均方误差损失函数
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam优化器

# 5. 训练模型
num_epochs = 100
for epoch in range(num_epochs):
    model.train()  # 设置模型为训练模式
    running_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()  # 清空梯度
        outputs = model(inputs)  # 前向传播
        loss = criterion(outputs.squeeze(), targets)  # 计算损失
        loss.backward()  # 反向传播
        optimizer.step()  # 更新参数
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

# 测试模型
model.eval()  # 设置模型为评估模式
predictions = []
targets = []

with torch.no_grad():  # 不计算梯度
    for inputs, target in test_loader:
        outputs = model(inputs).squeeze()  # 前向传播
        predictions.extend(scaler_y.inverse_transform(outputs.cpu().numpy().reshape(-1, 1)).flatten())  # 反标准化
        targets.extend(scaler_y.inverse_transform(target.cpu().numpy().reshape(-1, 1)).flatten())  # 反标准化

# 计算均方根误差 (RMSE)
import numpy as np
rmse = np.sqrt(np.mean((np.array(predictions) - np.array(targets))**2))
print(f"Test RMSE: {rmse:.4f}")

/home/xr/.conda/envs/d2l/lib/python3.10/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [1/100], Loss: 0.8643
Epoch [2/100], Loss: 0.6681
Epoch [3/100], Loss: 0.5126
Epoch [4/100], Loss: 0.5387
Epoch [5/100], Loss: 0.4697
Epoch [6/100], Loss: 0.4804
Epoch [7/100], Loss: 0.4463
Epoch [8/100], Loss: 0.4659
Epoch [9/100], Loss: 0.4738
Epoch [10/100], Loss: 0.4371
Epoch [11/100], Loss: 0.4352
Epoch [12/100], Loss: 0.4920
Epoch [13/100], Loss: 0.4949
Epoch [14/100], Loss: 0.4287
Epoch [15/100], Loss: 0.4146
Epoch [16/100], Loss: 0.4296
Epoch [17/100], Loss: 0.4084
Epoch [18/100], Loss: 0.3997
Epoch [19/100], Loss: 0.3966
Epoch [20/100], Loss: 0.3947
Epoch [21/100], Loss: 0.4216
Epoch [22/100], Loss: 0.3979
Epoch [23/100], Loss: 0.4569
Epoch [24/100], Loss: 0.4830
Epoch [25/100], Loss: 0.4216
Epoch [26/100], Loss: 0.3794
Epoch [27/100], Loss: 0.3983
Epoch [28/100], Loss: 0.3867
Epoch [29/100], Loss: 0.3741
Epoch [30/100], Loss: 0.3635
Epoch [31/100], Loss: 0.4234
Epoch [32/100], Loss: 0.4304
Epoch [33/100], Loss: 0.3571
Epoch [34/100], Loss: 0.3491
Epoch [35/100], Loss: 0